# Activity 2 — Adversarial Sensor Challenge

**Workshop:** From Black Boxes to Glass Boxes  
**Instructor:** Cagri Temel — Hezarfen LLC  
**Format:** Team competition · 3 to 4 students per team · 25 minutes

---

## The scenario

You are the data science team at an airline. Maintenance ops sends you three
suspicious test files for engine 17. In each file, **exactly one sensor channel
has been manipulated**. Your job: find which sensor and what kind of attack.

**Three attack types:**
- Drift — a constant offset added
- Stuck-at — frozen at a single value
- Gaussian noise — random noise injected

First team to identify all three correctly wins.

## How to use this notebook

There are **7 steps**. Each step has a markdown header and one empty code cell.
Paste from the slide, run with **Shift+Enter**.

**Important:** You will reuse the `tree`, `informative`, `scaler`, `X_train`, `y_train`,
`X_test`, `y_test`, and `labels` variables from Activity 1. Keep that notebook open in
another tab, or run the setup cell below first.


## Setup — install the library, download the files, re-train the model (run this first)

Activity 2 runs in a fresh Colab notebook, so it cannot see the variables you
trained in Activity 1. Run the cell below first. It will:

- Install `neural-trees`
- Download CMAPSS training data plus the three attack files
- Re-train the Soft Decision Tree so `tree`, `informative`, `scaler`, `labels`, and
  the train/test splits are all defined for the seven steps below

This takes about 60 seconds.


In [ ]:
# Workshop setup — run once before starting Step 1
!pip install -q neural-trees

import os
BASE = 'https://raw.githubusercontent.com/cgrtml/wsu-workshop-may15/main/data'
for f in ['train_FD001.txt', 'engine17_clean.csv',
          'attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from neural_trees import SoftDecisionTree

cols = ['engine_id', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
max_cycles = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.merge(max_cycles, on='engine_id')
train['RUL'] = (train['max_cycle'] - train['cycle']).clip(upper=125)

def bin_rul(r):
    if r < 30: return 0
    if r < 80: return 1
    return 2

train['health'] = train['RUL'].apply(bin_rul)
all_sensors = [f's{i}' for i in range(1, 22)]
informative = [s for s in all_sensors if train[s].std() > 0.001]
scaler = StandardScaler().fit(train[informative].values)
X = scaler.transform(train[informative].values)
y = train['health'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
tree = SoftDecisionTree(depth=4, max_epochs=30, learning_rate=0.01, penalty_coef=1e-3, verbose=False)
tree.fit(X_train, y_train)
labels = ['Critical', 'Caution', 'Healthy']
print('Setup done. tree, scaler, informative, labels are ready.')


## Step 1 — Add a RandomForest baseline

A black-box comparison for the explainable Soft Decision Tree. Expected: both models around 0.84 accuracy. Essentially identical on clean data.


In [ ]:
# Paste the code for Step 1 from the slide here, then Shift+Enter


## Step 2 — Predict on clean engine 17

Establish a reference: what do both models say about the un-attacked engine? Expected: 276 cycles, agreement around 0.95.


In [ ]:
# Paste the code for Step 2 from the slide here, then Shift+Enter


## Step 3 — How much did each attack change the predictions?

Run both models on each attack file and count cycle-by-cycle disagreement against the clean baseline. Watch Attack C carefully: the RandomForest barely notices it.


In [ ]:
# Paste the code for Step 3 from the slide here, then Shift+Enter


## Step 4 — Visualize Attack A (look for two parallel lines)

Plot every informative sensor twice — clean in blue, attack in orange. Fourteen panels overlap. One panel shows two separate lines: that is the manipulated sensor.


In [ ]:
# Paste the code for Step 4 from the slide here, then Shift+Enter


## Step 5 — Visualize Attack B (look for a flat horizontal line)

Same plot for attack B. The orange line on the manipulated sensor will be completely flat while the blue line keeps evolving.


In [ ]:
# Paste the code for Step 5 from the slide here, then Shift+Enter


## Step 6 — Visualize Attack C (look for a noisier line)

Attack C is subtler. The trend is preserved; only the variance changes. Look for the panel where orange is noticeably more jagged than blue.


In [ ]:
# Paste the code for Step 6 from the slide here, then Shift+Enter


## Step 7 — Quantify the divergence

Confirm the visual answer with numbers. For each attack file, the top-divergent sensor is the manipulated one. The other 14 sensors are identical between clean and attack.


In [ ]:
# Paste the code for Step 7 from the slide here, then Shift+Enter


---

## Your team's answers

Fill in the dictionary below before time is up. The instructor will collect.


In [ ]:
TEAM_NAME = ''  # e.g. 'Team Cougar'

answers = {
    'attack_A': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
    'attack_B': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
    'attack_C': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
}

print(f'Team: {TEAM_NAME}')
for k, v in answers.items():
    print(f'  {k}: sensor={v["sensor"]:4s}  type={v["type"]}')


---

## Wrap-up question

Discuss with your team:

> *In a real predictive-maintenance system, would you rely on the model's prediction, on its explanation, or both? When would you trust one over the other?*
